# §2. Model Training

In [181]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
    cross_val_score,
)
from sklearn.preprocessing import (
    PowerTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    LabelEncoder,
    FunctionTransformer,
)
from sklearn.impute import (
    SimpleImputer,
    KNNImputer,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.ensemble import (
    RandomForestClassifier,
)
import lightgbm as lgb
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.functional as F

from transformers import BertTokenizer, BertModel


import optuna

In [3]:
DATAPATH = Path("../data")
MODELPATH = Path("../models")


def read_data(filename, datapath=DATAPATH):
    file_path = datapath / filename
    if file_path.is_file():
        return pd.read_csv(file_path)
    else:
        raise FileNotFoundError(f"{file_path} does not exist.")


train_data = read_data("processed_train.csv")

In [32]:
chief_complaint_raw = train_data.loc[:, "chief_complaint_raw"]
max_length = chief_complaint_raw.str.len().max()
min_length = chief_complaint_raw.str.len().min()
mean_length = chief_complaint_raw.str.len().mean()
std_length = chief_complaint_raw.str.len().std()

print("statistics for chief_complaint_raw")

print("\nSequence length")
print(f"Max length: {max_length}")
print(f"Min length: {min_length}")
print(f"Mean length: {mean_length}")
print(f"Std length: {std_length}")


print("\nTokens")
tokens = chief_complaint_raw.str.split().apply(len)
print(f"Max tokens: {tokens.max()}")
print(f"Min tokens: {tokens.min()}")
print(f"Mean tokens: {tokens.mean()}")
print(f"Std tokens: {tokens.std()}")

statistics for chief_complaint_raw

Sequence length
Max length: 77
Min length: 7
Mean length: 39.6715125
Std length: 9.817311913528512

Tokens
Max tokens: 12
Min tokens: 1
Mean tokens: 5.3845375
Std tokens: 1.50880643798487


## § 2.1 Model Experiments

### § 2.1.1 Baseline Model (scikit-learn based)

In [165]:
# Baseline model:
# A scikit-learn model using structured features, then a scikit-learn pipeline 
# combining structured preprocessing with TF-IDF text features.

LABEL = 'triage_acuity'

physiological_variables = ['age', 'weight_kg', 'height_cm']
vital_signs_variables = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2']
other_variables = ['num_active_medications', 'num_comorbidities',  'gcs_total', 'pain_score', 'news2_score']
concurrency_variables = ['num_prior_ed_visits_12m', 'num_prior_admissions_12m']
categorical_variables = ['sex', 'arrival_mode', 'mental_status_triage', 'pain_location', 'chief_complaint_system', 'transport_origin']
hx_variables = ['hx_hypertension', 'hx_diabetes_type2',
       'hx_diabetes_type1', 'hx_asthma', 'hx_copd', 'hx_heart_failure',
       'hx_atrial_fibrillation', 'hx_ckd', 'hx_liver_disease', 'hx_malignancy',
       'hx_obesity', 'hx_depression', 'hx_anxiety', 'hx_dementia',
       'hx_epilepsy', 'hx_hypothyroidism', 'hx_hyperthyroidism', 'hx_hiv',
       'hx_coagulopathy', 'hx_immunosuppressed', 'hx_pregnant',
       'hx_substance_use_disorder', 'hx_coronary_artery_disease',
       'hx_stroke_prior', 'hx_peripheral_vascular_disease']
text_variables = ['chief_complaint_raw']

num_pipeline = Pipeline([
    ('imputer', KNNImputer()),
    ('scaler', PowerTransformer())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    # ('best', PCA(n_components=50, svd_solver='arpack'))
])

def mean_arterial_pressure(x):
    sbp, dbp = x[:, [0]], x[:, [1]]
    return dbp + 1/3*(sbp - dbp)

def pulse_pressure(x):
    sbp, dbp = x[:, [0]], x[:, [1]]
    return sbp - dbp

def bmi(x):
    weight = x[:, [0]]
    height = x[:, [1]]
    return (weight / height**2)*1e4

def shock_index(x):
    heart_rate = x[:, [0]]
    sbp = x[:, [1]]
    return heart_rate / sbp


mean_arterial_pressure_pipeline = make_pipeline(
    KNNImputer(),
    FunctionTransformer(mean_arterial_pressure)
)
pulse_pressure_pipeline = make_pipeline(
    KNNImputer(),
    FunctionTransformer(pulse_pressure)
)
bmi_pipeline = make_pipeline(
    KNNImputer(),
    FunctionTransformer(bmi)
)

shock_index_pipeline = make_pipeline(
    KNNImputer(),
    FunctionTransformer(shock_index)
)

preprocessor = ColumnTransformer([
    ('num', num_pipeline, physiological_variables + vital_signs_variables + other_variables + concurrency_variables),
    ('cat', cat_pipeline, categorical_variables),
    ('text', text_pipeline, *text_variables),
    ('map', mean_arterial_pressure_pipeline, ['systolic_bp', 'diastolic_bp']),
    ('pulse_pressure', pulse_pressure_pipeline, ['systolic_bp', 'diastolic_bp']),
    ('bmi', bmi_pipeline, ['weight_kg', 'height_cm']),
    ('shock_index', shock_index_pipeline, ['heart_rate', 'systolic_bp']),
    ('hx', 'passthrough', hx_variables)
    ],
    remainder='drop'
)

sk_model_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=1000))
])
sk_model_lgb = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', lgb.LGBMClassifier())
])


In [193]:
sample_train = train_data.sample(frac=0.1, random_state=42)
sample_train.shape

(8000, 52)

In [194]:
x_train = sample_train.drop(columns=[LABEL])
y_train = sample_train[LABEL]

In [168]:
preprocessor.fit(x_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

In [175]:
from sklearn.model_selection import (
    cross_val_score
)

lgb_score = cross_val_score(sk_model_lgb, x_train, y_train, cv=5)
lgb_score.mean()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003357 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10149
[LightGBM] [Info] Number of data points in the train set: 6400, number of used features: 412
[LightGBM] [Info] Start training from score -3.339103
[LightGBM] [Info] Start training from score -1.798029
[LightGBM] [Info] Start training from score -1.012578
[LightGBM] [Info] Start training from score -1.242735
[LightGBM] [Info] Start training from score -1.917110
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004227 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10125
[LightGBM] [Info] Number of data points in the train set: 6400, number of used features: 409
[LightGBM] 

np.float64(0.992625)

In [279]:
# training the model with full dataset

# croos entropy loss
from sklearn.metrics import log_loss

df = train_data.copy()

LABEL = 'triage_acuity'

cat_features = categorical_variables
num_features = (
    physiological_variables
    + vital_signs_variables
    + other_variables
    + concurrency_variables
    + hx_variables
)
text_col = text_variables[0]
features = num_features + cat_features + [text_col]


x_train, x_val, y_train, y_val = train_test_split(
    df[features],
    df[LABEL],
    test_size=0.2,
    random_state=42,
    stratify=df[LABEL],
)

baseline_model = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', lgb.LGBMClassifier())
])

baseline_model.fit(x_train, y_train)

# Predict probabilities once
proba_train = baseline_model.predict_proba(x_train)
proba_val = baseline_model.predict_proba(x_val)

# Keep class order explicit (important for multiclass log_loss)
classes = baseline_model.named_steps["clf"].classes_

# Log loss
train_loss = log_loss(y_train, proba_train, labels=classes)
val_loss = log_loss(y_val, proba_val, labels=classes)
print("Train Log Loss:", train_loss)
print("Validation Log Loss:", val_loss)

# Map class label -> probability column index
class_to_col = {c: i for i, c in enumerate(classes)}

train_idx = np.array([class_to_col[c] for c in y_train.to_numpy()])
val_idx = np.array([class_to_col[c] for c in y_val.to_numpy()])

train_true_prob = proba_train[np.arange(len(y_train)), train_idx].mean()
val_true_prob = proba_val[np.arange(len(y_val)), val_idx].mean()

print("Train avg true-class prob:", train_true_prob)
print("Validation avg true-class prob:", val_true_prob)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009659 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1994
[LightGBM] [Info] Number of data points in the train set: 64000, number of used features: 46
[LightGBM] [Info] Start training from score -3.211869
[LightGBM] [Info] Start training from score -1.783884
[LightGBM] [Info] Start training from score -1.017450
[LightGBM] [Info] Start training from score -1.245663
[LightGBM] [Info] Start training from score -1.948633
Train Log Loss: 0.2867706257622399
Validation Log Loss: 0.3611596230463372
Train avg true-class prob: 0.803509932480328
Validation avg true-class prob: 0.7767389569207467


In [281]:
# Save the trained model
import joblib

baseline_model_filename = "baseline_model.pkl"
joblib.dump(baseline_model, MODELPATH / baseline_model_filename)


['../models/baseline_model.pkl']

### § 2.1.2 transformer-based Model (Hugging Face)

In [268]:
tokenizer = BertTokenizer.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')
textModel = BertModel.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13123.27it/s]
[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [223]:
from typing import List
from transformers import PreTrainedTokenizer

class TriageDataset(Dataset):
    """Custom Dataset for Triage classification """
    def __init__(
            self,
            X_num: np.ndarray,
            X_cat: np.ndarray,
            X_text: np.ndarray,
            y: np.ndarray,
            tokenizer: PreTrainedTokenizer,
            max_length: int = 32
        ):
        """
        Initialize the TriageDataset
        
        Args:
            X_num (np.ndarray): Array with numerical features
            X_cat (np.ndarray): Array with categorical features
            X_text (np.ndarray): Array with text features
            y (np.ndarray): Target labels
            tokenizer (PreTrainedTokenizer): Tokenizer for text data
            max_length (int, optional): Maximum token length for text. Defaults to 32.
        """
        self.x_num = X_num.astype(np.float32)
        self.x_cat = X_cat.astype(np.int64)
        self.texts = X_text.tolist()
        self.y = y
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):        
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "x_num": torch.tensor(self.x_num[idx], dtype=torch.float32),
            "x_cat": torch.tensor(self.x_cat[idx], dtype=torch.long),
            "y": torch.tensor(self.y[idx], dtype=torch.long),
        }

In [224]:
LABEL = 'triage_acuity'

physiological_variables = ['age', 'weight_kg', 'height_cm']
vital_signs_variables = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2']
other_variables = ['num_active_medications', 'num_comorbidities',  'gcs_total', 'pain_score', 'news2_score']
concurrency_variables = ['num_prior_ed_visits_12m', 'num_prior_admissions_12m']
categorical_variables = ['sex', 'arrival_mode', 'mental_status_triage', 'pain_location', 'chief_complaint_system', 'transport_origin']
hx_variables = ['hx_hypertension', 'hx_diabetes_type2',
       'hx_diabetes_type1', 'hx_asthma', 'hx_copd', 'hx_heart_failure',
       'hx_atrial_fibrillation', 'hx_ckd', 'hx_liver_disease', 'hx_malignancy',
       'hx_obesity', 'hx_depression', 'hx_anxiety', 'hx_dementia',
       'hx_epilepsy', 'hx_hypothyroidism', 'hx_hyperthyroidism', 'hx_hiv',
       'hx_coagulopathy', 'hx_immunosuppressed', 'hx_pregnant',
       'hx_substance_use_disorder', 'hx_coronary_artery_disease',
       'hx_stroke_prior', 'hx_peripheral_vascular_disease'] # binary (0 or 1)
text_variables = ['chief_complaint_raw']

def make_preprocessor(num_features: List[str], cat_features: List[str]) -> ColumnTransformer:
    num_pipeline = Pipeline([
        ('imputer', KNNImputer()),
        ('scaler', PowerTransformer())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='UNK')),
        ('encoder', OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    text_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ])

    return ColumnTransformer(
        transformers=[
            ('num', num_pipeline, num_features),
            ('cat', cat_pipeline, cat_features),
            ],
        remainder='drop'
    )


def prepare_training_data(df: pd.DataFrame, tokenizer: PreTrainedTokenizer):
    """
    Prepare clinical data for training
    
    Args:
        df (pd.DataFrame): DataFrame with features and target columns
        tokenizer: ClinicalBERT tokenizer
    
    Returns:
        DataLoader: Training data loader
        DataLoader: Validation data loader
        ColumnTransformer: Preprocessor used for tabular data
        LabelEncoder: Label encoder for target labels
        List[int]: Cardinalities of categorical features        
    """
    cat_features = categorical_variables
    num_features = (
        physiological_variables
        + vital_signs_variables
        + other_variables
        + concurrency_variables
        + hx_variables
    )
    text_col = text_variables[0]
    features = num_features + cat_features + [text_col]

    x_train, x_val, y_train, y_val = train_test_split(
        df[features],
        df[LABEL],
        test_size=0.2,
        random_state=42,
        stratify=df[LABEL],
    )

    x_train_text = x_train[text_col].fillna("").astype(str).reset_index(drop=True)
    x_val_text = x_val[text_col].fillna("").astype(str).reset_index(drop=True)

    preprocessor = make_preprocessor(num_features, cat_features)
    x_train_tab = preprocessor.fit_transform(x_train)
    x_val_tab = preprocessor.transform(x_val)

    cat_cardinalities = [
        len(c) + 1 
        for c in preprocessor.named_transformers_['cat'].named_steps['encoder'].categories_
    ]

    n_num = len(num_features)
    n_cat = len(cat_features)

    x_train_num = x_train_tab[:, :n_num].astype(np.float32)
    x_val_num = x_val_tab[:, :n_num].astype(np.float32)

    # OrdinalEncoder gives -1 for unknown; shift by +1 so unknown becomes 0
    x_train_cat = (x_train_tab[:, n_num:n_num + n_cat].astype(np.int64) + 1)
    x_val_cat = (x_val_tab[:, n_num:n_num + n_cat].astype(np.int64) + 1)

    le = LabelEncoder()
    y_train = le.fit_transform(y_train)
    y_val = le.transform(y_val)
    
    train_dataset = TriageDataset(
        X_num=x_train_num,
        X_cat=x_train_cat,
        X_text=x_train_text,        
        y=y_train,
        tokenizer=tokenizer
    )
    val_dataset = TriageDataset(
        X_num=x_val_num,
        X_cat=x_val_cat,
        X_text=x_val_text,
        y=y_val,
        tokenizer=tokenizer
    )
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    return train_loader, val_loader, preprocessor, le, cat_cardinalities

In [252]:
# Build training and validation data loaders for the model
train_loader, val_loader, preprocessor, le, cat_cardinalities = prepare_training_data(sample_train, tokenizer)
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")

Number of training batches: 200
Number of validation batches: 50


In [253]:
for j, batch in enumerate(train_loader):
    print(f"index {j}")
    for key, value in batch.items():
        print(f"{key}: {value}")
    print("-----")
    break

index 0
input_ids: tensor([[  101,  7173, 20752,  ...,     0,     0,     0],
        [  101,  4910,  2332,  ...,     0,     0,     0],
        [  101,  6272, 10602,  ...,     0,     0,     0],
        ...,
        [  101,  1207, 13909,  ...,     0,     0,     0],
        [  101,  1892, 20636,  ...,     0,     0,     0],
        [  101,  1892, 20636,  ...,     0,     0,     0]])
attention_mask: tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])
x_num: tensor([[-0.8590,  1.2626,  2.2455,  ..., -0.6207,  1.6136, -0.5032],
        [ 0.8954,  0.1577, -0.1335,  ..., -0.6207, -0.6197, -0.5032],
        [ 0.3599, -0.2005,  0.1327,  ..., -0.6207, -0.6197, -0.5032],
        ...,
        [ 0.6690, -0.5325, -0.4783,  ..., -0.6207, -0.6197, -0.5032],
        [ 1.6573,  0.7322, -0.8226,  ...,  1.6111, -0.6197, -0.5032],
        [-0.546

In [254]:
class TriageModel(nn.Module):
    """Triage model combining text, numeric, and categorical features for classification."""
    def __init__(self, text_model, num_features, cat_cardinalities, n_classes):
        """Initialize the TriageModel.

        Args:
            text_model: Pretrained text model (e.g., BERT).
            num_features: Number of numeric features.
            cat_cardinalities: List of cardinalities for categorical features.
            n_classes: Number of output classes.
        """
        super().__init__()
        self.text_model = text_model
        hidden = text_model.config.hidden_size  # e.g., 768

        # categorical embeddings
        self.cat_embs = nn.ModuleList([
            nn.Embedding(cardinality, min(32, (cardinality + 1) // 2))
            for cardinality in cat_cardinalities
        ])
        cat_out = sum(emb.embedding_dim for emb in self.cat_embs)

        # numeric branch
        self.num_net = nn.Sequential(
            nn.LayerNorm(num_features),
            nn.Linear(num_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
        )

        fusion_in = hidden + 32 + cat_out
        self.head = nn.Sequential(
            nn.Linear(fusion_in, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, input_ids, attention_mask, x_num, x_cat):
        text_out = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_vec = text_out.last_hidden_state[:, 0, :]   # CLS

        cat_vecs = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embs)]
        cat_vec = torch.cat(cat_vecs, dim=1) if cat_vecs else None

        num_vec = self.num_net(x_num)

        if cat_vec is not None:
            fused = torch.cat([text_vec, num_vec, cat_vec], dim=1)
        else:
            fused = torch.cat([text_vec, num_vec], dim=1)

        return self.head(fused)

In [ ]:
import copy
from pathlib import Path

def train(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    num_epochs=20,
    patience=3,
    min_delta=1e-4,
    scheduler=None,
    checkpoint_path=None,
):
    model.to(device)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = {
        "train_loss": [],
        "val_loss": [],
        "lr": [],
    }

    for epoch in range(num_epochs):
        # ----- Train -----
        model.train()
        running_train_loss = 0.0

        for batch in train_loader:
            optimizer.zero_grad()

            y = batch["y"].to(device)
            x_num = batch["x_num"].to(device)
            x_cat = batch["x_cat"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(input_ids, attention_mask, x_num, x_cat)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * x_num.size(0)

        train_loss = running_train_loss / len(train_loader.dataset)

        # ----- Validate -----
        model.eval()
        running_val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                y = batch["y"].to(device)
                x_num = batch["x_num"].to(device)
                x_cat = batch["x_cat"].to(device)
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

                logits = model(input_ids, attention_mask, x_num, x_cat)
                loss = criterion(logits, y)
                running_val_loss += loss.item() * x_num.size(0)

        val_loss = running_val_loss / len(val_loader.dataset)

        # Scheduler step (ReduceLROnPlateau expects a metric)
        if scheduler is not None:
            scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["lr"].append(current_lr)

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {current_lr:.2e}"
        )

        # ----- Early stopping + best checkpoint -----
        improved = (best_val_loss - val_loss) > min_delta
        if improved:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0

            if checkpoint_path is not None:
                checkpoint_path = Path(checkpoint_path)
                checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
                torch.save(
                    {
                        "model_state_dict": best_state,
                        "best_val_loss": best_val_loss,
                    },
                    checkpoint_path,
                )
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}. Best val loss: {best_val_loss:.4f}")
                break

    # Restore best model weights before returning
    model.load_state_dict(best_state)

    return model, history, best_val_loss

def evaluate(model, data_loader, criterion, device):
    model.to(device)
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in data_loader:
            X_num = batch['x_num'].to(device)
            X_cat = batch['x_cat'].to(device)
            y = batch['y'].to(device)
            outputs = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device),
                X_num,
                X_cat
            )
            loss = criterion(outputs, y)
            total_loss += loss.item() * X_num.size(0)
    total_loss /= len(data_loader.dataset)
    return total_loss

In [274]:
num_features = train_loader.dataset.x_num.shape[1]
n_classes = len(le.classes_)

model_hyperparams = {
    "num_features": num_features,
    "cat_cardinalities": cat_cardinalities,
    "n_classes": n_classes
}

print("Model hyperparameters:", model_hyperparams)

Model hyperparameters: {'num_features': 41, 'cat_cardinalities': [4, 7, 6, 10, 15, 8], 'n_classes': 5}


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TriageModel(textModel, **model_hyperparams).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
)

best_ckpt = MODELPATH / "triage_best.pth"


In [ ]:
model, history, best_val = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=20,
    patience=3,
    min_delta=1e-4,
    scheduler=scheduler,
    checkpoint_path=best_ckpt,
)

print("Best val loss:", best_val)
# val_loss = evaluate(model, val_loader, criterion, device)
# print(f"Validation Loss after training: {val_loss:.4f}")

In [ ]:
# save the model for later use
import os

os.makedirs(MODELPATH, exist_ok=True)
model_data = {
    "model_state_dict": model.state_dict(), 
    "model_hyperparams": model_hyperparams
}
torch.save(model_data, MODELPATH / "triage_model.pth")

In [275]:
# load the model for later use
loaded_weights = torch.load(MODELPATH / "triage_model.pth", map_location=device)
triageModel = TriageModel(textModel, **loaded_weights["model_hyperparams"])
triageModel.load_state_dict(loaded_weights["model_state_dict"])
triageModel.eval()

TriageModel(
  (text_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, 

## References

https://markaicode.com/clinicalbert-medical-text-processing-guide/

https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer.html#sphx-glr-auto-examples-compose-plot-column-transformer-py

https://www.kaggle.com/code/jasonkarpeles/triagegeistmodel#Section-6:-Layer-3-%E2%80%94-LightGBM-Ensemble

https://www.kaggle.com/code/rahulchauhan016/emergency-department-triage-risk-prediction


